# 🎙️ Tetrax ASR: Fine-Tuning Whisper on AI4Bharat Svarah for Indian English

Welcome to the official **Tetrax ASR Fine-Tuning Notebook**.

This notebook trains and adapts **`openai/whisper-small`** (or `whisper-base`) using **PEFT / LoRA (Low-Rank Adaptation)** on the **AI4Bharat Svarah** dataset to build an Indian-accented English speech recognition model for Tetrax CPaaS.

### Why LoRA?
- Free Google Colab T4 GPU compatible (takes ~4GB VRAM).
- Only trains ~1.5% of parameters while freezing the core acoustic features.
- Saves adapter weights in seconds (~20-50MB instead of 1GB), making local deployment in Tetrax ASR instant.

---

## Step 1: Check GPU & Install Dependencies

In [ ]:
# Verify GPU allocation (T4, V100, or A100 recommended)
!nvidia-smi

# Install required fine-tuning libraries
!pip install -q transformers datasets peft accelerate evaluate jiwer soundfile librosa bitsandbytes tensorboard

## Step 2: (Optional) Mount Google Drive
Mount your drive to save checkpoints or load datasets directly so you never lose progress.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/Tetrax-ASR/checkpoints/tetrax-whisper-small-indian-en"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoint destination: {SAVE_DIR}")

## Step 3: Load AI4Bharat Svarah Dataset & Create Splits
This cell will:
1. Check if you uploaded `test-*.parquet` files to `/content/` or `/content/drive/MyDrive/Tetrax-ASR/data/`
2. If found, it loads directly from those files.
3. If not found, it automatically downloads from Hugging Face Hub (`ai4bharat/Svarah`).
4. Splits into a stratified 80% Train / 10% Validation / 10% Test split across Indian native languages.

In [ ]:
import glob
from datasets import load_dataset, Audio, DatasetDict

# Check for local parquet files
local_parquets = sorted(glob.glob('/content/test-*.parquet')) + sorted(glob.glob('/content/drive/MyDrive/Tetrax-ASR/data/test-*.parquet'))

if local_parquets:
    print(f"Found local parquet files: {local_parquets}")
    raw_dataset = load_dataset("parquet", data_files=local_parquets, split="train")
else:
    print("Loading ai4bharat/Svarah from Hugging Face Hub...")
    raw_dataset = load_dataset("ai4bharat/Svarah", split="test")

print(f"Total dataset samples: {len(raw_dataset)}")

# Use decode=False to ensure fast and robust loading
raw_dataset = raw_dataset.cast_column("audio_filepath", Audio(decode=False))

# Stratified 80/10/10 split across primary languages
stratify_col = None
if "primary_language" in raw_dataset.column_names:
    try:
        raw_dataset = raw_dataset.class_encode_column("primary_language")
        stratify_col = "primary_language"
    except Exception:
        stratify_col = None

first_split = raw_dataset.train_test_split(test_size=0.20, seed=42, stratify_by_column=stratify_col)
train_ds = first_split["train"]

second_split = first_split["test"].train_test_split(test_size=0.50, seed=42, stratify_by_column=stratify_col)
val_ds = second_split["train"]
test_ds = second_split["test"]

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(f"Train samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"Test samples: {len(test_ds)}")


### Inspect a Sample Audio & Transcription

In [ ]:
import io
import soundfile as sf
import IPython.display as ipd

sample = train_ds[0]
print("Transcript:", sample["text"])
print("Native Language:", sample.get("primary_language"))
print("State:", sample.get("native_place_state"))
print("Duration:", sample.get("duration"), "sec")

raw_bytes = sample["audio_filepath"]["bytes"]
audio_data, sr = sf.read(io.BytesIO(raw_bytes))
ipd.Audio(audio_data, rate=sr)

## Step 4: Setup Whisper Processor & Feature Extractor

In [ ]:
from transformers import WhisperProcessor

MODEL_ID = "openai/whisper-small"  # Or "openai/whisper-base"
print(f"Initializing processor for {MODEL_ID}...")

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="en", task="transcribe")

## Step 5: Preprocess Audio and Tokenize Text

In [ ]:
import librosa

def prepare_dataset(batch):
    # Load audio bytes with soundfile
    raw_bytes = batch["audio_filepath"]["bytes"]
    audio_arr, orig_sr = sf.read(io.BytesIO(raw_bytes), dtype="float32")
    if audio_arr.ndim > 1:
        audio_arr = audio_arr.mean(axis=1)
    if orig_sr != 16000:
        audio_arr = librosa.resample(audio_arr, orig_sr=orig_sr, target_sr=16000)
        
    # Compute 80-channel log-Mel spectrogram
    batch["input_features"] = processor.feature_extractor(
        audio_arr,
        sampling_rate=16000
    ).input_features[0]
    
    # Tokenize English transcript to label IDs
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

print("Mapping features and labels across splits...")
tokenized_train = dataset_dict["train"].map(prepare_dataset, remove_columns=dataset_dict["train"].column_names, num_proc=2)
tokenized_val = dataset_dict["validation"].map(prepare_dataset, remove_columns=dataset_dict["validation"].column_names, num_proc=2)
print("Preprocessing finished successfully!")

## Step 6: Data Collator and Evaluation Metric (WER)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## Step 7: Load Model & Apply PEFT LoRA Adapter

In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

print(f"Loading WhisperForConditionalGeneration ({MODEL_ID})...")
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# LoRA Configuration targeting attention projections
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 8: Fine-Tune with Seq2SeqTrainer

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import inspect

OUTPUT_DIR = "/content/tetrax-whisper-small-indian-en"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=100,
    num_train_epochs=3,
    eval_strategy="steps",
    fp16=True,
    per_device_eval_batch_size=8,
    generation_max_length=225,
    save_steps=100,
    eval_steps=100,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer_kwargs = {
    "args": training_args,
    "model": model,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_val,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

# Compatibility: transformers >= 4.46 / 5.x uses processing_class instead of tokenizer
if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
    trainer_kwargs["processing_class"] = processor.feature_extractor
else:
    trainer_kwargs["tokenizer"] = processor.feature_extractor

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Starting training loop...")
trainer.train()


## Step 9: Save Fine-Tuned Model & Export to Zip / Drive

In [ ]:
import shutil
import os
from google.colab import files

OUTPUT_DIR = "/content/tetrax-whisper-small-indian-en"
ZIP_NAME = "tetrax-whisper-small-indian-en.zip"

# 1. Save model & processor
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ Saved fine-tuned LoRA adapter to: {OUTPUT_DIR}")

# 2. Package into a compressed ZIP file
shutil.make_archive("tetrax-whisper-small-indian-en", "zip", OUTPUT_DIR)
zip_size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024) if os.path.exists(ZIP_NAME) else 0.0
print(f"📦 Successfully created ZIP archive: {ZIP_NAME} ({zip_size_mb:.2f} MB)")

# 3. Save both ZIP and uncompressed folder to Google Drive
DRIVE_DIR = "/content/drive/MyDrive/Tetrax-ASR/checkpoints"
try:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    drive_zip_path = os.path.join(DRIVE_DIR, ZIP_NAME)
    shutil.copy2(ZIP_NAME, drive_zip_path)
    print(f"💾 SAVED ZIP TO GOOGLE DRIVE: {drive_zip_path}")

    drive_folder_path = os.path.join(DRIVE_DIR, "tetrax-whisper-small-indian-en")
    shutil.copytree(OUTPUT_DIR, drive_folder_path, dirs_exist_ok=True)
    print(f"📁 SAVED FOLDER TO GOOGLE DRIVE: {drive_folder_path}")
except Exception as e:
    print(f"⚠️ Could not copy directly to Drive ({e}). Make sure drive.mount('/content/drive') was run in Step 2.")

# 4. Trigger direct browser download as local backup
print("⬇️ Triggering direct browser download of ZIP...")
try:
    files.download(ZIP_NAME)
except Exception as e:
    print("Browser download notice:", e)


## 🎉 Deployment into your Tetrax ASR Project

Once downloaded:
1. Extract `tetrax-whisper-small-indian-en.zip`.
2. Place the folder into `Tetrax-ASR/checkpoints/tetrax-whisper-small-indian-en/`.
3. Start your Tetrax ASR service:
   ```bash
   python -m uvicorn tetrax_asr.api.main:app --host 0.0.0.0 --port 8000
   ```
4. The Tetrax ASR engine automatically detects and mounts your fine-tuned Indian-English LoRA adapter for both REST API and real-time WebSocket live captions!